# Assignment 1: From Dirty Data to Predictive Models

## Titanic Survival Prediction

**Course:** Columbia AML  
**Task:** Predict whether a Titanic passenger survived (`Survived` ∈ {0, 1}).

This notebook develops an end-to-end supervised learning workflow covering:
1. Data Cleaning
2. Feature Engineering
3. Model Training (Naive Bayes + Linear Regression)
4. Model Evaluation
5. Discussion and AI Tool Usage Disclosure

> **Notebook goal:** Keep all code, outputs, visualizations, and technical reasoning needed for the final submission in this `.ipynb`.

## 0. Setup

This section imports the Python libraries that will be used throughout the assignment and defines a small number of constants for reproducibility.

The actual modeling dataset for this assignment is `train.csv`. The Kaggle `test.csv` and `gender_submission.csv` files are kept available but are not used for the assignment's model evaluation. We will create our own train/test split from the labeled `train.csv` data so that the actual `Survived` values are available for evaluation.

In [ ]:
# Core Python/data-analysis libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine-learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
)

# Reproducibility and model settings
RANDOM_STATE = 42
TEST_SIZE = 0.20
THRESHOLD = 0.5

# Plotting style
sns.set_theme(style="whitegrid")

print("Setup complete.")

In [ ]:
# The CSV files are uploaded to the Colab working directory.
TRAIN_PATH = "train.csv"
KAGGLE_TEST_PATH = "test.csv"

print(f"Training dataset: {TRAIN_PATH}")
print(f"Kaggle holdout (not used for assignment evaluation): {KAGGLE_TEST_PATH}")

### 0.1 Verify the uploaded files

Before loading the data, verify that the expected CSV files are available in the Colab environment.

In [ ]:
from pathlib import Path

available_files = sorted(p.name for p in Path(".").glob("*.csv"))
print("CSV files available in the current Colab directory:")
for file_name in available_files:
    print(f"- {file_name}")

assert Path(TRAIN_PATH).exists(), "train.csv was not found. Upload it to the Colab Files panel first."
print("\ntrain.csv found. Setup is ready for data exploration.")

In [ ]:
df = pd.read_csv(TRAIN_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
df.describe(include="all").T

In [ ]:
# Missingness overview
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")

In [ ]:
# Target balance
print(df["Survived"].value_counts())
print(df["Survived"].value_counts(normalize=True).round(3))
sns.countplot(data=df, x="Survived")
plt.title("Survival counts")
plt.show()

## 2. Data Cleaning

**TODO — justify in the report:**
- How you handle `Age`, `Cabin`, `Embarked` (and any other issues)
- Impute vs drop vs flag — why
- Any noisy / inconsistent values you fixed (e.g. `Fare`, ticket quirks)

In [ ]:
clean = df.copy()

# --- Example cleaning stubs (edit / replace with your justified choices) ---

# Embarked: few missing → fill with mode
embarked_mode = clean["Embarked"].mode()[0]
clean["Embarked"] = clean["Embarked"].fillna(embarked_mode)

# Age: impute with median (consider group median by Sex/Pclass later)
clean["Age"] = clean["Age"].fillna(clean["Age"].median())

# Cabin: high missingness → flag presence instead of raw cabin string
clean["HasCabin"] = clean["Cabin"].notna().astype(int)

# Optional: drop columns unused as raw features
# clean = clean.drop(columns=["Cabin", "Ticket", "Name", "PassengerId"])

clean.isna().sum()

## 3. Feature Engineering

**TODO:**
- Encode categoricals (`Sex`, `Embarked`, maybe `Pclass`)
- Scale continuous features as needed (esp. for NB / linear models)
- Optional new features: family size, title from `Name`, fare bins, etc.

In [ ]:
feat = clean.copy()

# --- Example engineered features ---
feat["FamilySize"] = feat["SibSp"] + feat["Parch"] + 1
feat["IsAlone"] = (feat["FamilySize"] == 1).astype(int)

# Title from Name (optional but useful)
feat["Title"] = feat["Name"].str.extract(r",\s*([^.]+)\.", expand=False)
# Collapse rare titles later if needed
print(feat["Title"].value_counts().head(10))

# Encode Sex
feat["Sex"] = feat["Sex"].map({"male": 0, "female": 1})

# One-hot Embarked (and optionally Title)
feat = pd.get_dummies(feat, columns=["Embarked"], drop_first=True)

feat.head()

In [ ]:
# Choose final model matrix
# Edit this list after you settle on features
feature_cols = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "HasCabin",
    "FamilySize",
    "IsAlone",
    # add Embarked_* dummies dynamically:
]
feature_cols += [c for c in feat.columns if c.startswith("Embarked_")]

X = feat[feature_cols].astype(float)
y = feat["Survived"].astype(int)

X.shape, y.shape

## 4. Train / test split

Use the **same split** for all models (fair comparison).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Scale continuous-ish features for models that benefit from it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train.shape, X_test.shape

## 5. Model Training

### 5a. Naive Bayes (generative)

Experiment with **≥ 2** `alpha` values (Laplace / add-α smoothing).  
Pick `GaussianNB` and/or `BernoulliNB` based on your feature types — justify the choice.

In [ ]:
def evaluate_classifier(name, y_true, y_pred, y_score=None):
    """Print metrics; return a small results dict."""
    cm = confusion_matrix(y_true, y_pred)
    results = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": cm,
    }
    if y_score is not None:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        results["auc"] = auc(fpr, tpr)
        results["roc"] = (fpr, tpr)
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, digits=3))
    print("Confusion matrix:\n", cm)
    if "auc" in results:
        print(f"AUC: {results['auc']:.3f}")
    return results


def plot_confusion(cm, title):
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# BernoulliNB with different alpha (Laplace smoothing)
# Note: BernoulliNB expects binary / boolean-ish features.
# If your matrix is mostly continuous, prefer GaussianNB below and
# discuss smoothing conceptually / try binarization.

alphas = [1.0, 0.01]  # required: try at least two
nb_results = []

# Simple binarization for BernoulliNB demo (edit as needed)
X_train_bin = (X_train > X_train.median()).astype(int)
X_test_bin = (X_test > X_train.median()).astype(int)

for alpha in alphas:
    nb = BernoulliNB(alpha=alpha)
    nb.fit(X_train_bin, y_train)
    y_pred = nb.predict(X_test_bin)
    y_score = nb.predict_proba(X_test_bin)[:, 1]
    res = evaluate_classifier(f"BernoulliNB(alpha={alpha})", y_test, y_pred, y_score)
    plot_confusion(res["confusion_matrix"], f"BernoulliNB α={alpha}")
    nb_results.append(res)

In [ ]:
# Optional: GaussianNB on scaled continuous features
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)
y_pred_gnb = gnb.predict(X_test_scaled)
y_score_gnb = gnb.predict_proba(X_test_scaled)[:, 1]
gnb_res = evaluate_classifier("GaussianNB", y_test, y_pred_gnb, y_score_gnb)
plot_confusion(gnb_res["confusion_matrix"], "GaussianNB")

### 5b. Linear Regression as binary classifier (discriminative)

Predict a continuous score, then classify with threshold `0.5`.  
Also try **Ridge (L2)** and **LASSO (L1)**.

In [ ]:
def fit_linear_as_classifier(model, name, X_tr, X_te):
    model.fit(X_tr, y_train)
    # clip scores to [0, 1] for nicer probability-like interpretation (optional)
    y_score = np.clip(model.predict(X_te), 0, 1)
    y_pred = (y_score >= THRESHOLD).astype(int)
    return evaluate_classifier(name, y_test, y_pred, y_score)


lin_results = []

lin_results.append(
    fit_linear_as_classifier(
        LinearRegression(), "LinearRegression", X_train_scaled, X_test_scaled
    )
)
plot_confusion(lin_results[-1]["confusion_matrix"], "LinearRegression")

lin_results.append(
    fit_linear_as_classifier(
        Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "Ridge(alpha=1.0)",
        X_train_scaled,
        X_test_scaled,
    )
)
plot_confusion(lin_results[-1]["confusion_matrix"], "Ridge")

lin_results.append(
    fit_linear_as_classifier(
        Lasso(alpha=0.01, random_state=RANDOM_STATE, max_iter=10000),
        "Lasso(alpha=0.01)",
        X_train_scaled,
        X_test_scaled,
    )
)
plot_confusion(lin_results[-1]["confusion_matrix"], "LASSO")

## 6. Model Evaluation & comparison

Required: accuracy + confusion matrix  
Encouraged: precision / recall / F1  
Bonus: ROC + AUC  

**Discuss in report:** effect of smoothing (`alpha`) vs weaker / stronger smoothing.

In [ ]:
all_results = nb_results + [gnb_res] + lin_results

summary = pd.DataFrame(
    [
        {
            "model": r["model"],
            "accuracy": r["accuracy"],
            "precision": r["precision"],
            "recall": r["recall"],
            "f1": r["f1"],
            "auc": r.get("auc", np.nan),
        }
        for r in all_results
    ]
).sort_values("accuracy", ascending=False)

summary

In [ ]:
# Bonus: ROC curves
plt.figure(figsize=(6, 5))
for r in all_results:
    if "roc" in r:
        fpr, tpr = r["roc"]
        plt.plot(fpr, tpr, label=f"{r['model']} (AUC={r['auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="chance")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC curves")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Scratch notes for the PDF report

Fill these before writing the 10–12 page report (no code in the PDF):

- **Cleaning decisions:** …
- **Feature choices:** …
- **NB alpha comparison takeaway:** …
- **Linear vs NB:** …
- **Limitations:** …
- **AI disclosure:** tools used, what they contributed, what you did yourself

In [ ]:
# Optional: Kaggle submission file from ../data/test.csv
# (Not required by the assignment; only if you want a leaderboard score.)
#
# kaggle_test = pd.read_csv(TEST_PATH)
# ... apply the same cleaning + feature pipeline ...
# preds = (best_model.predict(X_kaggle) >= THRESHOLD).astype(int)
# submission = pd.DataFrame({"PassengerId": kaggle_test["PassengerId"], "Survived": preds})
# submission.to_csv("../data/submission.csv", index=False)